In [1]:
import os
# 打印/kaggle/input下全部一级文件夹
print("/kaggle/input 下所有已挂载文件夹：")
root_input = "/kaggle/input"
folder_list = os.listdir(root_input)
for folder_name in folder_list:
    full_path = os.path.join(root_input, folder_name)
    print(f"文件夹名称：{folder_name} | 完整路径：{full_path}")

/kaggle/input 下所有已挂载文件夹：
文件夹名称：competitions | 完整路径：/kaggle/input/competitions


In [ ]:
import os
comp_root = "/kaggle/input/competitions"
sub_folders = os.listdir(comp_root)
for name in sub_folders:
    full = os.path.join(comp_root, name)
    print(f"竞赛子文件夹名：{name} | 完整路径：{full}")

In [ ]:
import os
# 这里替换成上面代码打印出来的competitions内部文件夹名
COMPETITION_FOLDER_NAME = "bh-2026"

# 完整数据根路径
DATA_ROOT = os.path.join("/kaggle/input", "competitions", COMPETITION_FOLDER_NAME, "img_AIdet", "data")
TRAIN_REAL = os.path.join(DATA_ROOT, "train", "REAL")
TRAIN_FAKE = os.path.join(DATA_ROOT, "train", "FAKE")
TEST_DIR = os.path.join(DATA_ROOT, "test")

# 校验
check_paths = [DATA_ROOT, TRAIN_REAL, TRAIN_FAKE, TEST_DIR]
for p in check_paths:
    print(f"{p} 存在？ {os.path.exists(p)}")

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np

# 已验证可用的绝对路径
DATA_ROOT = "/kaggle/input/competitions/bh-2026/img_AIdet/data"
TRAIN_REAL = os.path.join(DATA_ROOT, "train", "REAL")  # 真图 label=0
TRAIN_FAKE = os.path.join(DATA_ROOT, "train", "FAKE")  # 假图 label=1
TEST_DIR = os.path.join(DATA_ROOT, "test")

# 全局参数
IMG_SIZE = 32
BATCH_SIZE = 512  # 期望输出shape是[512,3,32,32]
VAL_SPLIT_RATIO = 0.1
SPLIT_SEED = 42  # 固定种子保证划分可复现

In [ ]:
class TrainAIDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.samples = []
        # 真图：label=0
        for fname in sorted(os.listdir(real_dir)):
            if fname.endswith((".jpg", ".jpeg", ".png")):
                self.samples.append((os.path.join(real_dir, fname), 0))
        # 假图：label=1
        for fname in sorted(os.listdir(fake_dir)):
            if fname.endswith((".jpg", ".jpeg", ".png")):
                self.samples.append((os.path.join(fake_dir, fname), 1))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label

In [ ]:
class TestAIDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.transform = transform
        self.test_dir = test_dir
        # 严格按1.jpg~20000.jpg排序
        self.file_list = [(i, f"{i}.jpg") for i in range(1, 20001)]
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        img_id, fname = self.file_list[idx]
        img = Image.open(os.path.join(self.test_dir, fname)).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, img_id

In [ ]:
# 训练集增强：RandomCrop+水平翻转
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 8, IMG_SIZE + 8)),  # 适配padding=4的RandomCrop
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证/测试集：仅ToTensor+Normalize，无增强
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# 构建完整训练集
full_train_dataset = TrainAIDataset(TRAIN_REAL, TRAIN_FAKE, transform=train_transform)

# 固定随机种子划分9:1
generator = torch.Generator().manual_seed(SPLIT_SEED)
val_size = int(len(full_train_dataset) * VAL_SPLIT_RATIO)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

# 验证集要替换成无增强的transform
val_dataset.dataset.transform = val_test_transform

# 测试集
test_dataset = TestAIDataset(TEST_DIR, transform=val_test_transform)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0  # Kaggle环境设0避免多进程报错
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [ ]:
# 验证shape是否符合期望 [512,3,32,32]
for x, y in train_loader:
    print("训练批次shape:", x.shape, y.shape)
    assert x.shape == torch.Size([512, 3, 32, 32]), "输出shape不符合要求！"
    break

In [ ]:
print(f"训练集长度（应≈90000）: {len(train_dataset)}")
print(f"验证集长度（应≈10000）: {len(val_dataset)}")
print(f"测试集长度（必须=20000）: {len(test_dataset)}")
assert len(test_dataset) == 20000, "测试集数量不对！"

In [ ]:
all_labels = set()
for _, labels in train_loader:
    all_labels.update(labels.numpy().tolist())
for _, labels in val_loader:
    all_labels.update(labels.numpy().tolist())
print("所有标签取值:", all_labels)
assert all_labels == {0, 1}, "标签只能是0和1！"

In [ ]:
import random
import matplotlib.pyplot as plt

# 实例化数据集（无transform，原图）
dataset_raw = TrainAIDataset(TRAIN_REAL, TRAIN_FAKE, transform=None)
real_count = len(os.listdir(TRAIN_REAL))

# 随机抽一张真实图（0 ~ real_count-1）
rand_real_idx = random.randint(0, real_count - 1)
real_img, _ = dataset_raw[rand_real_idx]
plt.figure()
plt.imshow(real_img)
plt.title(f"Real Image, label=0 | Index:{rand_real_idx}")
plt.show()

# 随机抽一张AI假图（real_count ~ len(dataset_raw)-1）
rand_fake_idx = random.randint(real_count, len(dataset_raw)-1)
fake_img, _ = dataset_raw[rand_fake_idx]
plt.figure()
plt.imshow(fake_img)
plt.title(f"AI Fake Image, label=1 | Index:{rand_fake_idx}")
plt.show()

In [ ]:
# ========== 阶段1验收专用：样本溯源核验（双图对比+文件路径打印） ==========
import os
import random
from PIL import Image
import matplotlib.pyplot as plt

# 固定数据集路径（已验证可用）
DATA_ROOT = "/kaggle/input/competitions/bh-2026/img_AIdet/data"
TRAIN_REAL = os.path.join(DATA_ROOT, "train", "REAL")
TRAIN_FAKE = os.path.join(DATA_ROOT, "train", "FAKE")

# 数据集类（和你之前完全一致，保证读取顺序统一）
class TrainAIDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.samples = []
        # 先加载全部真实图 label=0
        for fname in sorted(os.listdir(real_dir)):
            if fname.endswith(".jpg"):
                self.samples.append((os.path.join(real_dir, fname), 0))
        # 后加载全部假图 label=1
        for fname in sorted(os.listdir(fake_dir)):
            if fname.endswith(".jpg"):
                self.samples.append((os.path.join(fake_dir, fname), 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label

# 实例化无增强原始数据集
raw_dataset = TrainAIDataset(TRAIN_REAL, TRAIN_FAKE, transform=None)
real_total = len(os.listdir(TRAIN_REAL))
fake_total = len(os.listdir(TRAIN_FAKE))
print(f"REAL total count: {real_total} | FAKE total count: {fake_total}")
print(f"Total training samples: {len(raw_dataset)}")

# 随机抽取REAL区间下标 0 ~ real_total-1
rand_real_idx = random.randint(0, real_total - 1)
read_img, label = raw_dataset[rand_real_idx]

# 找到该下标对应的原始文件
real_file_list = sorted(os.listdir(TRAIN_REAL))
target_file = real_file_list[rand_real_idx]
target_path = os.path.join(TRAIN_REAL, target_file)
origin_img = Image.open(target_path).convert("RGB")

# 打印溯源信息
print("===== Real Image Verification Info =====")
print(f"Global dataset index: {rand_real_idx}")
print(f"Corresponding file name in folder: {target_file}")
print(f"Full file path: {target_path}")
print(f"Sample label: {label}")

# 左右并排对比：代码读取图 VS 原始文件图
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(read_img)
plt.title(f"Dataset Loaded Image | Index:{rand_real_idx}, label={label}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(origin_img)
plt.title(f"Original File | {target_file}")
plt.axis("off")
plt.show()

# 随机抽取FAKE区间下标 real_total ~ len(raw_dataset)-1
rand_fake_idx = random.randint(real_total, len(raw_dataset) - 1)
read_img, label = raw_dataset[rand_fake_idx]

# 计算fake内部下标，找到原始文件
fake_inner_idx = rand_fake_idx - real_total
fake_file_list = sorted(os.listdir(TRAIN_FAKE))
target_file = fake_file_list[fake_inner_idx]
target_path = os.path.join(TRAIN_FAKE, target_file)
origin_img = Image.open(target_path).convert("RGB")

# 打印溯源信息
print("===== AI Fake Image Verification Info =====")
print(f"Global dataset index: {rand_fake_idx}")
print(f"Inner index inside FAKE folder: {fake_inner_idx}")
print(f"Corresponding file name in folder: {target_file}")
print(f"Full file path: {target_path}")
print(f"Sample label: {label}")

# 左右并排对比
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(read_img)
plt.title(f"Dataset Loaded Image | Index:{rand_fake_idx}, label={label}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(origin_img)
plt.title(f"Original File | {target_file}")
plt.axis("off")
plt.show()

In [5]:
%%writefile train_pipeline.py
# ========== 数据处理精简交付脚本（仅模型训练使用，无绘图/校验冗余代码） ==========
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# 自动区分Kaggle云端 / 本地运行环境
if os.path.exists("/kaggle/input"):
    DATA_ROOT = "/kaggle/input/competitions/bh-2026/img_AIdet/data"
else:
    # 本地运行时仅需修改此处根目录
    DATA_ROOT = "D:/bh-2026/img_AIdet/data"

# 全局固定超参
TRAIN_REAL = os.path.join(DATA_ROOT, "train", "REAL")
TRAIN_FAKE = os.path.join(DATA_ROOT, "train", "FAKE")
TEST_DIR = os.path.join(DATA_ROOT, "test")
IMG_SIZE = 32
BATCH_SIZE = 512
SPLIT_SEED = 42

# 训练集数据集读取类
class TrainAIDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.samples = []
        # 优先加载真实图片，标签0
        for fname in sorted(os.listdir(real_dir)):
            if fname.endswith(".jpg"):
                file_path = os.path.join(real_dir, fname)
                self.samples.append((file_path, 0))
        # 后加载AI假图，标签1
        for fname in sorted(os.listdir(fake_dir)):
            if fname.endswith(".jpg"):
                file_path = os.path.join(fake_dir, fname)
                self.samples.append((file_path, 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label

# 测试集数据集读取类（输出图片ID用于生成提交csv）
class TestAIDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.transform = transform
        self.test_dir = test_dir
        # 测试图片固定编号1~20000
        self.file_list = [(i, f"{i}.jpg") for i in range(1, 20001)]

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_id, fname = self.file_list[idx]
        img_path = os.path.join(self.test_dir, fname)
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, img_id

# 训练集专属数据增强（严格遵循作业要求）
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 8, IMG_SIZE + 8)),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证集、测试集变换：无随机增强，仅标准化缩放
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 加载完整训练数据集
full_train_dataset = TrainAIDataset(TRAIN_REAL, TRAIN_FAKE, train_transform)
# 固定seed拆分9:1训练/验证集
split_generator = torch.Generator().manual_seed(SPLIT_SEED)
val_sample_num = int(len(full_train_dataset) * 0.1)
train_sample_num = len(full_train_dataset) - val_sample_num
train_dataset, val_dataset = random_split(full_train_dataset, [train_sample_num, val_sample_num], generator=split_generator)
# 验证集替换为无增强transform
val_dataset.dataset.transform = val_test_transform

# 加载测试集
test_dataset = TestAIDataset(TEST_DIR, val_test_transform)

# 批量加载器（适配Kaggle num_workers=0，本地可自行修改为2/4加速）
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Writing train_pipeline.py


In [6]:
import os
print(os.path.exists("/kaggle/working/train_pipeline.py"))

True


In [7]:
exec(open("/kaggle/working/train_pipeline.py").read())
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
x, y = next(iter(train_loader))
print(f"Batch shape: {x.shape}")

Train: 90000, Val: 10000, Test: 20000
Batch shape: torch.Size([512, 3, 32, 32])
